# Quant Skills — Lesson 6: Market Making

**Goal:** build a working market maker, watch it make money and lose money, and fix it the way real desks do.

You'll run four versions, each fixing the previous one's biggest problem:

1. **Naive** — quote both sides, collect the spread
2. **Inventory control** — stop getting stuck with a big position
3. **Informed traders arrive** — the thing that actually kills market makers
4. **Adaptive spreads** — widen up when the risk goes up

**How to run it:** **Runtime → Run all**. No downloads — it's a simulated market. Lines you can edit are marked `# <- EDIT`.

## Step 1 — Load the tools

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("Tools loaded.")

## Step 2 — How market making works

A market maker always shows two prices: a **bid** (where they'll buy) and an **ask** (where they'll sell).
If someone sells to them at the bid and someone else buys from them at the ask, they pocket the difference.

**Two things can go wrong:**
- **Inventory risk** — you buy, then nobody buys from you, and the price falls while you're holding.
- **Adverse selection** — the people trading with you sometimes *know* where price is going. They buy from you right before it rises.

Every rule in professional market making exists to manage those two risks.

## Step 3 — The simulated market

In [ ]:
SIZE        = 100      # shares per fill
SIGMA       = 0.02     # typical price move per step ($)
STEPS_DAY   = 2_000    # steps in one trading day
DAYS        = 20       # <- EDIT: trading days to simulate
A, K        = 0.12, 60 # how often traders arrive, and how fast that drops as your quote moves away from fair value
HORIZON     = 4        # informed traders know where price will be this many steps ahead
FEE         = 0.0      # <- EDIT: exchange fee per share per fill (negative = rebate)

def simulate(half_spread=0.01, skew=0.0, max_inv=10**9, informed=0.0,
             adaptive=False, seed=1):
    """Run one market maker on a simulated market. Returns a dict of results."""
    rng = np.random.default_rng(seed)
    n = STEPS_DAY * DAYS
    # volatility regimes: some stretches are calm, some are wild
    regime = np.repeat(rng.choice([0.6, 1.0, 2.2], size=n // 400 + 1, p=[0.4, 0.45, 0.15]), 400)[:n + HORIZON]
    moves = rng.normal(0, SIGMA, n + HORIZON) * regime
    mid = 100 + np.concatenate([[0], np.cumsum(moves)])
    U = rng.random((n, 3))

    cash, inv, spread_pnl, fills = 0.0, 0, 0.0, 0
    inv_path, pnl_path = np.zeros(n), np.zeros(n)
    for t in range(n):
        s, hs = mid[t], half_spread
        if adaptive and t > 50:
            vol_now = np.sqrt(np.mean(moves[t - 50:t] ** 2))
            hs = half_spread * max(1.0, (vol_now / SIGMA) ** 2)   # widen hard when the market gets wild
        r = s - skew * inv / SIZE                                   # reservation price: shade quotes against inventory
        bid, ask = r - hs, r + hs
        u1, u2, u3 = U[t]

        # informed traders: show up more in wild markets, and only trade when they'll beat your spread
        if informed > 0 and u3 < informed * regime[t]:
            future = mid[t + HORIZON] - s
            if future > ask - s and inv > -max_inv:
                cash += (ask - FEE) * SIZE; inv -= SIZE; fills += 1; spread_pnl += (ask - s) * SIZE
            elif future < -(s - bid) and inv < max_inv:
                cash -= (bid + FEE) * SIZE; inv += SIZE; fills += 1; spread_pnl += (s - bid) * SIZE

        # ordinary traders: more likely to trade with you the closer your quote is to fair value
        if u1 < min(1.0, A * np.exp(-K * (ask - s))) and inv > -max_inv:
            cash += (ask - FEE) * SIZE; inv -= SIZE; fills += 1; spread_pnl += (ask - s) * SIZE
        if u2 < min(1.0, A * np.exp(-K * (s - bid))) and inv < max_inv:
            cash -= (bid + FEE) * SIZE; inv += SIZE; fills += 1; spread_pnl += (s - bid) * SIZE

        inv_path[t] = inv
        pnl_path[t] = cash + inv * mid[t + 1]

    daily = np.diff(np.concatenate([[0], pnl_path[STEPS_DAY - 1::STEPS_DAY]]))
    fees = -FEE * SIZE * fills
    return {"pnl": pnl_path, "inv": inv_path, "daily": daily, "fills": fills,
            "spread_pnl": spread_pnl, "inventory_pnl": pnl_path[-1] - spread_pnl - fees,
            "fees": fees, "mid": mid}

def report(name, res):
    d = res["daily"]
    sharpe = d.mean() / d.std() * np.sqrt(252) if d.std() > 0 else float("nan")
    print(f"{name}")
    print(f"   Total P&L:            ${res['pnl'][-1]:>9,.0f}   over {DAYS} days")
    print(f"   Average day:          ${d.mean():>9,.0f}   ·  worst day ${d.min():,.0f}")
    print(f"   Sharpe (annualized):  {sharpe:>9.1f}")
    print(f"   Earned from spread:   ${res['spread_pnl']:>9,.0f}")
    print(f"   Inventory gain/loss:  ${res['inventory_pnl']:>9,.0f}")
    if res["fees"]:
        print(f"   Fees:                 ${res['fees']:>9,.0f}")
    print(f"   Largest position:     {int(np.abs(res['inv']).max()):>9,} shares")
    print(f"   Fills:                {res['fills']:>9,}")
    print()

print("Market ready.")

## Step 4 — Version 1: the naive market maker

Quote 1 cent either side of fair value, no matter what. No inventory limit.

In [ ]:
naive = simulate(half_spread=0.01)
report("1. NAIVE", naive)

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(naive["pnl"], linewidth=1); ax[0].set_ylabel("P&L ($)"); ax[0].grid(alpha=0.3)
ax[0].set_title("Naive market maker")
ax[1].plot(naive["inv"], linewidth=1, color="tab:orange"); ax[1].set_ylabel("Inventory (shares)")
ax[1].set_xlabel("Time step"); ax[1].grid(alpha=0.3); plt.show()

**Look at the split between the two numbers.** *Earned from spread* is the actual market-making business.
*Inventory gain/loss* is something else: the position wanders thousands of shares in one direction, so the P&L
becomes a bet on where price goes. If that number is large — positive or negative — this run's result was mostly
luck. Step 8 runs it 30 times so you can see the bad-luck version too.

## Step 5 — Version 2: inventory control

Two fixes real desks use:
- **Skew** — when you're long, lower *both* your quotes. Your ask becomes more attractive (so you sell more)
  and your bid less attractive (so you buy less). The position works itself back toward zero.
- **A hard limit** — stop buying past a maximum position.

In [ ]:
SKEW    = 0.002   # <- EDIT: how hard to shade quotes per 100 shares of inventory
MAX_INV = 1_000   # <- EDIT: hard position limit in shares

controlled = simulate(half_spread=0.01, skew=SKEW, max_inv=MAX_INV)
report("2. INVENTORY CONTROL", controlled)

plt.figure(figsize=(10, 3.5))
plt.plot(naive["inv"], linewidth=0.8, alpha=0.6, label="Naive")
plt.plot(controlled["inv"], linewidth=0.8, label="With skew + limit")
plt.ylabel("Inventory (shares)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Inventory stays near zero with skew"); plt.show()

## Step 6 — Version 3: informed traders arrive

Now some of the people trading with you know where price is going next. This is **adverse selection** —
and it's the reason real market makers lose money.

In [ ]:
INFORMED = 0.05   # <- EDIT: how often an informed trader shows up (0.05 = 5% of steps in a normal market)

hit = simulate(half_spread=0.01, skew=SKEW, max_inv=MAX_INV, informed=INFORMED)
report("3. INVENTORY CONTROL + INFORMED TRADERS", hit)

**Notice:** the spread still earns money — often *more*, because informed traders trade with you a lot. But the
inventory line turns sharply negative: every time one hits you, price moves against the position you just took.
A strategy that made money every single run now loses money. That's the "pick off" — and it's the main reason
real market makers fail.

## Step 7 — Version 4: adaptive spreads

The fix: **widen your quotes when the market gets more volatile**, because that's when informed traders do the most
damage. Same skew, same limit — just a spread that breathes with conditions.

In [ ]:
WIDE_HALF = 0.02   # <- EDIT: base half-spread for the adaptive version

adaptive = simulate(half_spread=WIDE_HALF, skew=SKEW, max_inv=MAX_INV, informed=INFORMED, adaptive=True)
report("4. ADAPTIVE SPREADS", adaptive)

plt.figure(figsize=(10, 4.5))
for name, res in [("1. Naive", naive), ("2. Inventory control", controlled),
                  ("3. + Informed traders", hit), ("4. Adaptive spreads", adaptive)]:
    plt.plot(res["pnl"], linewidth=1.2, label=name)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("All four versions"); plt.ylabel("P&L ($)"); plt.xlabel("Time step")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Step 8 — Is it skill or luck? Run it 30 times

One run can be lucky. Run each version on 30 different random markets and compare the averages.
This is the same Monte Carlo idea from Lesson 1.

In [ ]:
RUNS = 30   # <- EDIT: more runs = more reliable, slower

versions = {
    "Naive":                     dict(half_spread=0.01),
    "Inventory control":         dict(half_spread=0.01, skew=SKEW, max_inv=MAX_INV),
    "+ Informed traders":        dict(half_spread=0.01, skew=SKEW, max_inv=MAX_INV, informed=INFORMED),
    "Adaptive spreads":          dict(half_spread=WIDE_HALF, skew=SKEW, max_inv=MAX_INV, informed=INFORMED, adaptive=True),
}
for name, kw in versions.items():
    totals = np.array([simulate(seed=s, **kw)["pnl"][-1] for s in range(RUNS)])
    print(f"{name:22s}  average ${totals.mean():>9,.0f}   worst ${totals.min():>9,.0f}   "
          f"profitable in {(totals > 0).mean():.0%} of runs")

## Step 9 — Your exercises

1. **Step 5:** set `SKEW = 0`. What happens to the largest position and the inventory P&L?
2. **Step 6:** try `INFORMED` at `0.02`, then `0.10`. Find roughly where the market maker stops making money.
3. **Step 7:** try `WIDE_HALF` at `0.015`, `0.02`, `0.03`. Wider spreads mean fewer fills but fewer pick-offs — find the sweet spot.
4. **Step 3:** set `FEE = 0.01` — a 1-cent fee per share, the same size as the spread — and run all. This is why retail
   market making on Coinbase didn't work: when fees are as big as your edge, there's nothing left.

## Turning this into your portfolio project

This notebook — plus the Python market maker we built earlier — is the core of a project you can show a firm.
A strong write-up covers:

| Section | What to show |
|---|---|
| **The problem** | Quote two-sided markets while controlling inventory and adverse selection |
| **The model** | Reservation price with inventory skew, hard limits, volatility-adaptive spreads |
| **Results** | The four-version comparison and the 30-run Monte Carlo from Step 8 |
| **What you learned** | Where the edge comes from (spread) and where it leaks (inventory, informed flow, fees) |
| **Real-world test** | Kalshi or crypto data recorded with your recorder, run through the same logic |

Put it on GitHub with a short README and the charts. That's evidence of skill no résumé line can match.

**Course complete.** You've covered expected value, volatility, options pricing, fat tails, backtesting, and market making —
the core of what a junior quant is expected to know.